In [1]:
import numpy as np
from math import sqrt
from pprint import pprint

from sklearn import datasets, linear_model, metrics
from sklearn.neighbors import KNeighborsRegressor        # regresión con K vecinos más cercanos
from sklearn.preprocessing import StandardScaler         # estandarización de features
from sklearn.feature_selection import SelectPercentile, f_regression  # selección de características
from sklearn.metrics import make_scorer
from sklearn.model_selection import (
    cross_val_predict,   # predicciones por validación cruzada
    cross_val_score,     # puntuación por validación cruzada
    cross_validate,      # validación cruzada con múltiples métricas
    train_test_split,    # partición hold-out
    KFold                # partición en K subconjuntos
)
from sklearn import preprocessing

import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")  # suprimir warnings de sklearn

In [7]:
# Carga de datos.
dataset = datasets.fetch_california_housing()
X = dataset.data
y = dataset.target
print(np.shape(X))

(20640, 8)


In [6]:
# DEFINICIÓN DE MÉTRICAS DE EVALUACIÓN

metricas = {
    # MAE (Mean Absolute Error): error absoluto medio
    # ya está integrada en sklearn, se referencia por nombre
    'MAE': 'neg_mean_absolute_error',

    # RMSE (Root Mean Squared Error): raíz del error cuadrático medio
    # no está directamente en sklearn → se calcula con lambda
    # greater_is_better=False → indica que menor valor = mejor modelo
    'RMSE': make_scorer(lambda y, y_pred:
                sqrt(metrics.mean_squared_error(y, y_pred)),
                greater_is_better=False),

    # MAPE (Mean Absolute Percentage Error): error porcentual absoluto medio
    # mide el error en % respecto al valor real → más interpretable
    # greater_is_better=False → menor % = mejor modelo
    'MAPE': make_scorer(lambda y, y_pred:
                np.mean(np.abs((y - y_pred) / y)) * 100,
                greater_is_better=False)
}

### 1) PARTICIÓN EXTERNA DE DATOS

In [8]:
# 1) PARTICIÓN HOLD-OUT

# Divide el dataset en train (80%) y test (20%)
# random_state=42 → resultados reproducibles
X_training, X_testing, y_training, y_testing = train_test_split(X, y, test_size=0.2, random_state=42)

print(np.shape(X_training))  # → (16512, 8) → 80% de las muestras, 8 features
print(np.shape(X_testing))   # → (4128, 8)  → 20% de las muestras, 8 features

(16512, 8)
(4128, 8)


### 2-5) ENTRENAMIENTO

In [ ]:
# 2) Extracción de características
# 3) Selección de atributos

In [9]:
# 4) ESTANDARIZACIÓN DE LOS DATOS DE ENTRENAMIENTO

standardizer = preprocessing.StandardScaler()

# fit_transform() = fit() + transform() en una sola línea
# Aprende la media y std de X_training y aplica la estandarización al mismo tiempo
# Equivalente al código anterior:
#   stdr_trained = standardizer.fit(X_training)
#   X_stdr = stdr_trained.transform(X_training)
X_stdr = standardizer.fit_transform(X_training)

fit_transform solo se usa con training. Para el test debes usar solo transform

In [12]:
# 5) CONSTRUCCIÓN DEL MODELO KNN PARA REGRESIÓN

k = 10  # número de vecinos a considerar

reg = KNeighborsRegressor(
    n_neighbors = k,          # usa los 10 vecinos más cercanos para predecir
    weights     = 'distance', # vecinos más cercanos tienen más peso en la predicción
    metric      = 'euclidean' # distancia euclídea para medir similitud entre puntos
)

In [13]:
# 5.1) VALIDACIÓN CRUZADA INTERNA

# cross_val_score evalúa el modelo con validación cruzada
# Parámetros:
#   reg          → modelo KNN definido en el paso anterior
#   X_stdr       → datos de entrenamiento estandarizados
#   y_training   → etiquetas de entrenamiento
#   cv           → KFold con 5 particiones, aleatorias y semilla 42
#   scoring      → métrica R² (coeficiente de determinación, 1=perfecto, 0=malo)
r2_cv_results = cross_val_score(reg, X_stdr, y_training,
                                cv=KFold(n_splits=5, shuffle=True, random_state=42),
                                scoring='r2')

print("cross_val_R2:   %0.4f +/- %0.4f" % (r2_cv_results.mean(), r2_cv_results.std()))

cross_val_R2:   0.6962 +/- 0.0096


In [ ]:
# Extraer métricas MAE, MSE, RMSE y MAPE en un cross validation para 5 bolsas aleatorias y semilla en 42
metrics_cv_results = ???
pprint(metrics_cv_results)

In [ ]:
# Extraer las predicciones del cross validation de 5 bolsas aleatorias y semilla en 42
y_pred = ???

In [ ]:
# Crear una función que dadas las variables "y" e "y_pred" se visualice la bisectriz
def plot_bisectriz(y, y_pred):
    fig, ax = plt.subplots()
    ax.scatter(y, y_pred, edgecolors=(0, 0, 0))
    ax.plot([y.min(), y.max()], [y.min(), y.max()], 'k--', lw=4)
    ax.set_xlabel('Valor real de la clase')
    ax.set_ylabel('Predicción')
    plt.grid()
    plt.show()

# Visualiza la bisetriz
???

In [ ]:
# 5.2) Entrenamiento del modelo definitivo
model = reg.fit(X_stdr, y_training)

### 6-10) PREDICCIÓN

In [ ]:
# 6) Extracción de las características de test
# 7) Selección de los atributos de test

In [ ]:
# 8) Estandarización de las característiacs de test
X_test_stdr = standardizer.transform(X_testing)

In [ ]:
# 9) Predicción del conjunto de test
y_pred_test = model.predict(X_test_stdr)

In [ ]:
# 10) Evaluación del modelo sobre el conjunto de test
MAE = metrics.mean_absolute_error(y_testing, y_pred_test)
MSE = metrics.mean_squared_error(y_testing, y_pred_test, squared=True)
RMSE = metrics.mean_squared_error(y_testing, y_pred_test, squared=False)
R2 = metrics.r2_score(y_testing, y_pred_test)

print('MAE:  %.4f' % MAE)
print('MSE: %.4f' % MSE)
print('RMSE: %.4f' % RMSE)
print('R2:   %.4f' % R2)

# Visualización de resultados
plot_bisectriz(y_testing, y_pred_test)

### ACTIVIDAD EXTRA

#### Entrenar el mejor modelo posible con el algoritmo KNN y comparar el resultado con el modelo OLS en test

In [ ]:
from sklearn.model_selection import GridSearchCV
# Definir parámetros de búsqueda
???

# Aplicar el algoritmo de búsqueda
???

# Entrenar el modelo
???

# Extraer los mejores parámetros
???

In [ ]:
# Entrenar cada algoritmo y comparar los resultados (a nivel cuantitativo y cualitativo) sobre el conjunto de test
???